# Fine-tuning - Prédiction de gravité des ACCIDENTS

## Contexte

Ce notebook fait suite au notebook **3-a-accident-initial-ml-training** qui a établi :
- La baseline de performance avec RandomForest et XGBoost
- La comparaison Binary vs Ordered (choix du target binaire)
- L'analyse des feature importances

## Objectif

Optimiser les hyperparamètres pour améliorer les performances au-delà de la baseline.

## Workflow

```
1. Configuration et imports
2. Chargement des données
3. GridSearchCV (recherche des meilleurs hyperparamètres)
4. Impact du nombre d'arbres (n_estimators)
5. Comparaison des algorithmes (RF, XGBoost, LightGBM, CatBoost)
6. Sélection finale et sauvegarde
```

## Critères de sélection

| Critère | Métrique | Justification |
|---------|----------|---------------|
| **Principal** | F1-Score | Équilibre précision/rappel |
| **Secondaire** | AUC | Départage en cas d'égalité |

---

## 1. Configuration et imports

In [1]:
# === IMPORTS ===
import os
import pandas as pd
import numpy as np
import joblib

from functions.models import evaluate_model, evaluate_catboost, create_pipeline
from functions.data_preparation import prepare_data
from ml_config import nb_workers, base_estimators, max_evals
from functions.data_load import load_dataset
from functions import (
    display_metrics,
    optimize_boosting_model,
    plot_optimization_history,
    select_best_model,
    save_best_model,
)
from functions.mlflow_tracking import init_mlflow, log_training_run, log_hyperopt_run, log_final_model
from ml_config import MLFLOW_EXPERIMENT_TUNING

# Visualisation
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc,
)
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

# === CONFIGURATION ===
# Seed pour reproductibilité : garantit les mêmes résultats à chaque exécution
RANDOM_STATE = 42
print("Imports OK")

d:\Suivi_de_formation\simplon-ai-developer-training\W31-W33-ML-DEPLOYMENT-CI-CD\mlflow-EDUCATIONAL\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Imports OK


In [2]:
# === MLflow ===
mlflow_ready = init_mlflow(MLFLOW_EXPERIMENT_TUNING)

d:\Suivi_de_formation\simplon-ai-developer-training\W31-W33-ML-DEPLOYMENT-CI-CD\mlflow-EDUCATIONAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow connecte a http://localhost:5000 (experiment: tuning)


---

## 2. Chargement des données et fonctions utilitaires

In [3]:
# Chargement du dataset ACCIDENT
df_accident = load_dataset("dataset_accident")

# Aperçu des données
print(f"\n📊 Colonnes disponibles ({len(df_accident.columns)}):")
print(df_accident.columns.tolist())

dataset_accident: chargé depuis Parquet (263,356 lignes, 11 colonnes)

📊 Colonnes disponibles (11):
['grav_ordered', 'grav_binary', 'est_nuit', 'est_heure_pointe', 'jour_semaine', 'est_weekend', 'agg', 'vma', 'impl_vehicule_leger', 'impl_poids_lourd', 'impl_pieton']


## 3. Hyperopt - Optimisation bayésienne des hyperparamètres

### Pourquoi Hyperopt plutôt que GridSearchCV ?

| Aspect | GridSearchCV | Hyperopt (TPE) |
|--------|--------------|----------------|
| **Méthode** | Exhaustive (teste tout) | Bayésienne (apprend des essais) |
| **Efficacité** | O(n^k) combinaisons | Converge plus vite |
| **Espace** | Grille discrète | Continu + discret |
| **Intelligence** | Aucune | Mémorise les bons/mauvais essais |

### Algorithme TPE (Tree-structured Parzen Estimator)

1. Évalue quelques points aléatoires
2. Sépare les résultats en "bons" et "mauvais"
3. Modélise P(params | bons) et P(params | mauvais)
4. Choisit le prochain point qui maximise le ratio

In [4]:
# === HYPEROPT - OPTIMISATION BAYÉSIENNE ===

# Préparation des données
X_acc_bin, y_acc_bin = prepare_data(df_accident, "grav_binary")
X_train_acc_bin, X_test_acc_bin, y_train_acc_bin, y_test_acc_bin = train_test_split(
    X_acc_bin, y_acc_bin, test_size=0.2, random_state=RANDOM_STATE, stratify=y_acc_bin
)

# Sous-échantillonnage pour accélérer (50k)
sample_size_acc = min(50000, len(X_train_acc_bin))
X_sample_acc = X_train_acc_bin.sample(n=sample_size_acc, random_state=RANDOM_STATE)
y_sample_acc = y_train_acc_bin.loc[X_sample_acc.index]

print(f"Hyperopt sur {sample_size_acc:,} echantillons")
print("=" * 50)

# === RANDOMFOREST ===
best_params_rf, best_f1_rf, trials_rf = optimize_boosting_model(
    X_sample_acc,
    y_sample_acc,
    model_type="randomforest",
    max_evals=max_evals,
    cv=3,
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=nb_workers,
)

# Validation sur test set complet
print("\nValidation sur test set complet...")
best_model_rf = create_pipeline(
    RandomForestClassifier(**best_params_rf, random_state=RANDOM_STATE, n_jobs=nb_workers, class_weight="balanced")
)
best_model_rf.fit(X_train_acc_bin, y_train_acc_bin)
y_pred_rf = best_model_rf.predict(X_test_acc_bin)
f1_rf_test = f1_score(y_test_acc_bin, y_pred_rf)
print(f"F1 sur test set: {f1_rf_test:.3f}")

Hyperopt sur 50,000 echantillons
Optimisation Hyperopt pour RANDOMFOREST...
  - max_evals: 50
  - cv: 3 folds
  - scoring: f1

100%|██████████| 50/50 [09:58<00:00, 11.96s/trial, best loss: -0.5438478406898531]

Logging 50 trials Hyperopt dans MLflow...
🏃 View run hyperopt_randomforest_#001 at: http://localhost:5000/#/experiments/3/runs/eea72430126c4826bab7495d1c01151c
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_randomforest_#002 at: http://localhost:5000/#/experiments/3/runs/bb6a768dfd8e47f08003141427454d10
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_randomforest_#003 at: http://localhost:5000/#/experiments/3/runs/8a39aace35d24d6d921aad770fa2b8f4
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_randomforest_#004 at: http://localhost:5000/#/experiments/3/runs/68012c2108bd4529bfbd535ed0c0ec03
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_randomforest_#0

### 3.2 Hyperopt - CatBoost

CatBoost utilise des hyperparametres differents de RandomForest :
- **iterations** (n_estimators) : nombre d'arbres (attention a l'overfitting)
- **learning_rate** : pas d'apprentissage (plus petit = plus lent mais plus precis)
- **depth** : profondeur max (typiquement 4-10 pour boosting)
- **l2_leaf_reg** : regularisation L2

In [5]:
# === HYPEROPT - CATBOOST ===

best_params_cb, best_f1_cb, trials_cb = optimize_boosting_model(
    X_sample_acc,
    y_sample_acc,
    model_type="catboost",
    max_evals=max_evals,
    cv=3,
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=nb_workers,
)

# Validation sur test set complet
print("\nValidation sur test set complet...")
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_acc_bin), columns=X_train_acc_bin.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_acc_bin), columns=X_test_acc_bin.columns)

best_model_cb = CatBoostClassifier(
    **best_params_cb, random_state=RANDOM_STATE, auto_class_weights="Balanced", verbose=False, allow_writing_files=False
)
best_model_cb.fit(X_train_imp, y_train_acc_bin)
y_pred_cb = best_model_cb.predict(X_test_imp)
f1_cb_test = f1_score(y_test_acc_bin, y_pred_cb)
print(f"F1 sur test set: {f1_cb_test:.3f}")

Optimisation Hyperopt pour CATBOOST...
  - max_evals: 50
  - cv: 3 folds
  - scoring: f1

100%|██████████| 50/50 [08:58<00:00, 10.78s/trial, best loss: -0.5464853412643697]

Logging 50 trials Hyperopt dans MLflow...
🏃 View run hyperopt_catboost_#001 at: http://localhost:5000/#/experiments/3/runs/e3515c62ceff40d5aa94ac2b6967b4df
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_catboost_#002 at: http://localhost:5000/#/experiments/3/runs/2f0f5d0e928b4877a9ccce8bf02dc64a
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_catboost_#003 at: http://localhost:5000/#/experiments/3/runs/3a57787bf68b4905b6b6b396c08a2b92
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_catboost_#004 at: http://localhost:5000/#/experiments/3/runs/4a8b91adb74f49579bab3f3069014c48
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_catboost_#005 at: http://localhost:5000/#/experiments/3/runs/c6acfeb

### 3.3 Hyperopt - XGBoost

XGBoost (eXtreme Gradient Boosting) utilise des hyperparamètres similaires à CatBoost :
- **n_estimators** : nombre d'arbres
- **learning_rate** : taux d'apprentissage
- **max_depth** : profondeur max des arbres
- **min_child_weight** : poids minimum par feuille (régularisation)
- **subsample** / **colsample_bytree** : échantillonnage pour réduire l'overfitting

In [6]:
best_params_xgb, best_f1_xgb, trials_xgb = optimize_boosting_model(
    X_sample_acc,
    y_sample_acc,
    model_type="xgboost",
    max_evals=max_evals,
    cv=3,
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=nb_workers,
)

# Validation sur test set complet
print("\nValidation sur test set complet...")
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_acc_bin), columns=X_train_acc_bin.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_acc_bin), columns=X_test_acc_bin.columns)

best_model_xgb = XGBClassifier(
    **best_params_xgb,
    random_state=RANDOM_STATE,
    scale_pos_weight=18 / 10,
    verbosity=0,
)
best_model_xgb.fit(X_train_imp, y_train_acc_bin)
y_pred_xgb = best_model_xgb.predict(X_test_imp)
f1_xgb_test = f1_score(y_test_acc_bin, y_pred_xgb)
print(f"F1 sur test set: {f1_xgb_test:.3f}")

Optimisation Hyperopt pour XGBOOST...
  - max_evals: 50
  - cv: 3 folds
  - scoring: f1

100%|██████████| 50/50 [02:29<00:00,  2.99s/trial, best loss: -0.5478081904090369]

Logging 50 trials Hyperopt dans MLflow...
🏃 View run hyperopt_xgboost_#001 at: http://localhost:5000/#/experiments/3/runs/3244d24959174a27b12c3ad4c12db40a
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_xgboost_#002 at: http://localhost:5000/#/experiments/3/runs/f52275b4871d4769b3f307843d57678f
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_xgboost_#003 at: http://localhost:5000/#/experiments/3/runs/45c1c2a580a140f0a7f205f12e8e3212
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_xgboost_#004 at: http://localhost:5000/#/experiments/3/runs/bc6ec48dc29743ff88b9bc04ba7025bf
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_xgboost_#005 at: http://localhost:5000/#/experiments/3/runs/c9c1f571a0b94

### 3.4 Hyperopt - LightGBM

LightGBM (Light Gradient Boosting Machine) est optimisé pour la vitesse :
- **n_estimators** : nombre d'arbres
- **learning_rate** : taux d'apprentissage
- **max_depth** : profondeur max (peut être -1 pour illimité)
- **num_leaves** : nombre max de feuilles par arbre (spécifique à LightGBM)
- **min_child_samples** : échantillons minimum par feuille

In [7]:
best_params_lgbm, best_f1_lgbm, trials_lgbm = optimize_boosting_model(
    X_sample_acc,
    y_sample_acc,
    model_type="lightgbm",
    max_evals=max_evals,
    cv=3,
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=nb_workers,
)

# Validation sur test set complet
print("\nValidation sur test set complet...")
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_acc_bin), columns=X_train_acc_bin.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_acc_bin), columns=X_test_acc_bin.columns)

best_model_lgbm = LGBMClassifier(**best_params_lgbm, random_state=RANDOM_STATE, class_weights="balanced", verbose=-1)
best_model_lgbm.fit(X_train_imp, y_train_acc_bin)
y_pred_lgbm = best_model_lgbm.predict(X_test_imp)
f1_lgbm_test = f1_score(y_test_acc_bin, y_pred_xgb)
print(f"F1 sur test set: {f1_lgbm_test:.3f}")

Optimisation Hyperopt pour LIGHTGBM...
  - max_evals: 50
  - cv: 3 folds
  - scoring: f1

100%|██████████| 50/50 [01:46<00:00,  2.12s/trial, best loss: -0.4826539586613931] 

Logging 50 trials Hyperopt dans MLflow...
🏃 View run hyperopt_lightgbm_#001 at: http://localhost:5000/#/experiments/3/runs/9bbf9bac405243ffb936f7b1f42d78a5
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_lightgbm_#002 at: http://localhost:5000/#/experiments/3/runs/8ef4a4482eb8405587fa13dfedb4afab
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_lightgbm_#003 at: http://localhost:5000/#/experiments/3/runs/f6a90db9fd2d4aaf96a568d7393cf141
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_lightgbm_#004 at: http://localhost:5000/#/experiments/3/runs/f656ce3f22a14cf79f82776bb9e14ee2
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run hyperopt_lightgbm_#005 at: http://localhost:5000/#/experiments/3/runs/5e89e5

### 3.5 Convergence de l'optimisation

Visualisation de la progression d'Hyperopt pour chaque algorithme.

**Interprétation :**
- Les points représentent chaque essai (combinaison d'hyperparamètres)
- La courbe rouge montre le meilleur score cumulé
- Une courbe qui se stabilise indique que l'espace a été bien exploré

In [8]:
# === VISUALISATION DE LA CONVERGENCE HYPEROPT ===

fig = make_subplots(rows=1, cols=2, subplot_titles=["RandomForest", "CatBoost"])

# RF
scores_rf = [t["result"]["score"] for t in trials_rf.trials]
best_rf = np.maximum.accumulate(scores_rf)
fig.add_trace(go.Scatter(y=scores_rf, mode="markers", name="RF essais", marker=dict(size=5, opacity=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(y=best_rf, mode="lines", name="RF best", line=dict(color="red", width=2)), row=1, col=1)

# CatBoost
scores_cb = [t["result"]["score"] for t in trials_cb.trials]
best_cb = np.maximum.accumulate(scores_cb)
fig.add_trace(go.Scatter(y=scores_cb, mode="markers", name="CB essais", marker=dict(size=5, opacity=0.5)), row=1, col=2)
fig.add_trace(go.Scatter(y=best_cb, mode="lines", name="CB best", line=dict(color="red", width=2)), row=1, col=2)

# XGBoost
scores_xgb = [t["result"]["score"] for t in trials_xgb.trials]
best_xgb = np.maximum.accumulate(scores_xgb)
fig.add_trace(
    go.Scatter(y=scores_xgb, mode="markers", name="xgb essais", marker=dict(size=5, opacity=0.5)), row=1, col=2
)
fig.add_trace(go.Scatter(y=best_xgb, mode="lines", name="xgb best", line=dict(color="red", width=2)), row=1, col=2)

# LightGBM
scores_lgbm = [t["result"]["score"] for t in trials_lgbm.trials]
best_lgbm = np.maximum.accumulate(scores_lgbm)
fig.add_trace(
    go.Scatter(y=scores_lgbm, mode="markers", name="lgbm essais", marker=dict(size=5, opacity=0.5)), row=1, col=2
)
fig.add_trace(go.Scatter(y=best_lgbm, mode="lines", name="lgbm best", line=dict(color="red", width=2)), row=1, col=2)

fig.update_layout(title="Convergence Hyperopt", height=400, width=900, showlegend=False)
fig.update_xaxes(title_text="Essai")
fig.update_yaxes(title_text="F1 Score")
fig.show()

# Résumé
print("\n" + "=" * 50)
print("RESUME HYPEROPT")
print("=" * 50)
print(f"RandomForest : F1 CV={best_f1_rf:.3f} | F1 test={f1_rf_test:.3f}")
print(f"CatBoost     : F1 CV={best_f1_cb:.3f} | F1 test={f1_cb_test:.3f}")
print(f"XGBoost      : F1 CV={best_f1_xgb:.3f} | F1 test={f1_xgb_test:.3f}")
print(f"LightGBM     : F1 CV={best_f1_lgbm:.3f} | F1 test={f1_lgbm_test:.3f}")


RESUME HYPEROPT
RandomForest : F1 CV=0.544 | F1 test=0.525
CatBoost     : F1 CV=0.546 | F1 test=0.544
XGBoost      : F1 CV=0.548 | F1 test=0.538
LightGBM     : F1 CV=0.483 | F1 test=0.538


---

## 8. GridSearchCV + MLflow

### Recherche exhaustive d'hyperparametres

GridSearchCV teste **toutes les combinaisons** d'une grille predefinies. Chaque combinaison est loguee comme un **run MLflow distinct**, ce qui permet de comparer visuellement dans l'UI.

| Aspect | GridSearchCV | Hyperopt (TPE) |
|--------|--------------|----------------|
| **Methode** | Exhaustive (teste tout) | Bayesienne (apprend) |
| **Avantage** | Garanti de trouver le meilleur | Plus rapide |
| **Inconvenient** | O(n^k) combinaisons | Peut rater le global |
| **Logging MLflow** | Manuel (iteration sur cv_results_) | Manuel |

**Attention** : On utilise une grille **reduite** pour limiter le temps de calcul.

In [9]:
# === ETAPE 8 : GridSearchCV + MLflow ===
from functions.gridsearch_tuning import run_gridsearch_with_mlflow

# Grille reduite pour XGBoost (3 x 3 x 3 = 27 combinaisons)
param_grid_xgb = {
    "max_depth": [3, 5, 7],
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2],
}

# Lancer GridSearchCV sur le sous-echantillon (meme que Hyperopt)
# Les runs sont logues dans l'experiment "tuning" (deja initialisee)
grid_xgb, best_params_grid, best_score_grid = run_gridsearch_with_mlflow(
    X_sample_acc,
    y_sample_acc,
    model_type="xgboost",
    param_grid=param_grid_xgb,
    cv=3,
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=nb_workers,
)

GridSearchCV pour XGBOOST...
  - 27 combinaisons a tester
  - cv: 3 folds
  - scoring: f1

Fitting 3 folds for each of 27 candidates, totalling 81 fits

Logging 27 runs dans MLflow...
🏃 View run gridsearch_xgboost_#001_rank27 at: http://localhost:5000/#/experiments/3/runs/1b9dfaa9465f4315b9f3f26b2ace5afb
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run gridsearch_xgboost_#002_rank26 at: http://localhost:5000/#/experiments/3/runs/ffdf72da95da45dc91ea6a5e95d50455
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run gridsearch_xgboost_#003_rank25 at: http://localhost:5000/#/experiments/3/runs/5a41c8f4d97f4cada4c835b9e28ccbf4
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run gridsearch_xgboost_#004_rank24 at: http://localhost:5000/#/experiments/3/runs/1ba4a9d001304358afb1074ef46d9625
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run gridsearch_xgboost_#005_rank22 at: http://localhost:5000/#/experiments/3/runs

In [10]:
# === Visualisation des resultats GridSearchCV ===
results_df = pd.DataFrame(grid_xgb.cv_results_)
results_df = results_df.sort_values("rank_test_score")

# Top 10 des meilleures combinaisons
print("Top 10 des combinaisons GridSearchCV (XGBoost):")
print(results_df[["params", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]].head(10).to_string())

# Heatmap : F1 score en fonction de max_depth et learning_rate (pour n_estimators optimal)
best_n_est = best_params_grid["n_estimators"]
subset = results_df[results_df["param_n_estimators"] == best_n_est]

pivot = subset.pivot_table(
    values="mean_test_score",
    index="param_max_depth",
    columns="param_learning_rate",
)

fig_heatmap = go.Figure(
    data=go.Heatmap(
        z=pivot.values,
        x=[f"lr={c}" for c in pivot.columns],
        y=[f"depth={r}" for r in pivot.index],
        text=np.round(pivot.values, 4),
        texttemplate="%{text}",
        colorscale="Viridis",
    )
)
fig_heatmap.update_layout(
    title=f"GridSearchCV - F1 Score (n_estimators={best_n_est})",
    xaxis_title="Learning Rate",
    yaxis_title="Max Depth",
    width=600,
    height=400,
)
fig_heatmap.show()

print(f"\nComparaison avec Hyperopt XGBoost:")
print(f"  GridSearchCV best F1 (CV): {best_score_grid:.4f}")
print(f"  Hyperopt best F1 (CV):     {best_f1_xgb:.4f}")

Top 10 des combinaisons GridSearchCV (XGBoost):
                                                         params  mean_test_score  std_test_score  mean_train_score  rank_test_score
19  {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200}         0.545291        0.002212          0.555615                1
14  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}         0.545283        0.002283          0.561010                2
12  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}         0.545050        0.001611          0.558334                3
22  {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200}         0.545026        0.002797          0.561645                4
21  {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100}         0.544947        0.002712          0.560349                5
20  {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 300}         0.544914        0.002577          0.555688                6
24  {'learning_rate': 0.2, '


Comparaison avec Hyperopt XGBoost:
  GridSearchCV best F1 (CV): 0.5453
  Hyperopt best F1 (CV):     0.5478


---

## 9. Optuna + MLflow

### Recherche bayesienne moderne

Optuna est un framework d'optimisation bayesienne plus recent et plus flexible que Hyperopt. Il offre :
- **MLflowCallback** natif : chaque trial est automatiquement logue
- **Visualisations integrees** : importance des parametres, historique, coordonnees paralleles
- **Pruning** : arret anticipie des essais non prometteurs (non utilise ici)

| Aspect | GridSearchCV | Hyperopt | Optuna |
|--------|-------------|----------|--------|
| **Methode** | Exhaustive | TPE bayesien | TPE bayesien |
| **API** | sklearn native | Fonctionnelle | Orientee objet |
| **MLflow** | Manuel | Manuel | Callback natif |
| **Visualisations** | Aucune | Basiques | Riches (Plotly) |
| **Pruning** | Non | Non | Oui |

In [11]:
# === ETAPE 9 : Optuna + MLflow ===
from functions.optuna_tuning import run_optuna_with_mlflow, plot_optuna_results

# Lancer Optuna sur XGBoost (meme sous-echantillon, meme nombre d'essais que Hyperopt)
# Les runs sont logues dans l'experiment "tuning" (deja initialisee)
study_xgb, best_params_optuna, best_score_optuna = run_optuna_with_mlflow(
    X_sample_acc,
    y_sample_acc,
    model_type="xgboost",
    n_trials=max_evals,
    cv=3,
    scoring="f1",
    random_state=RANDOM_STATE,
    n_jobs=nb_workers,
)

[I 2026-02-25 11:41:34,787] A new study created in memory with name: optuna_xgboost


Optimisation Optuna pour XGBOOST...
  - n_trials: 50
  - cv: 3 folds
  - scoring: f1

  MLflow callback non disponible: 
Could not find `optuna-integration` for `mlflow`.
Please run `pip install optuna-integration[mlflow]`.


Best trial: 0. Best value: 0.542717:   2%|▏         | 1/50 [00:03<02:30,  3.08s/it]

[I 2026-02-25 11:41:37,864] Trial 0 finished with value: 0.542717235679522 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481}. Best is trial 0 with value: 0.542717235679522.


Best trial: 1. Best value: 0.543897:   4%|▍         | 2/50 [00:04<01:41,  2.11s/it]

...

[I 2026-02-25 11:43:53,389] Trial 47 finished with value: 0.5410490934894626 and parameters: {'n_estimators': 250, 'max_depth': 9, 'learning_rate': 0.20539792701128884, 'min_child_weight': 6, 'subsample': 0.7989810112599784, 'colsample_bytree': 0.9644936995025688}. Best is trial 39 with value: 0.5484243487656777.


Best trial: 39. Best value: 0.548424:  98%|█████████▊| 49/50 [02:20<00:02,  2.25s/it]

[I 2026-02-25 11:43:55,139] Trial 48 finished with value: 0.542683936924004 and parameters: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.07543241891124033, 'min_child_weight': 5, 'subsample': 0.7352642025672337, 'colsample_bytree': 0.9877858705536665}. Best is trial 39 with value: 0.5484243487656777.


Best trial: 39. Best value: 0.548424: 100%|██████████| 50/50 [02:22<00:00,  2.85s/it]

[I 2026-02-25 11:43:57,269] Trial 49 finished with value: 0.5448885645775721 and parameters: {'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.059508614471715376, 'min_child_weight': 6, 'subsample': 0.7082040837041262, 'colsample_bytree': 0.9996398745337757}. Best is trial 39 with value: 0.5484243487656777.

Meilleurs parametres XGBOOST:
  - n_estimators: 250
  - max_depth: 3
  - learning_rate: 0.15523941207697325
  - min_child_weight: 6
  - subsample: 0.7351449578299228
  - colsample_bytree: 0.7137268621791049

Meilleur f1 (CV): 0.5484


In [12]:
# === Visualisations Optuna ===
# Historique d'optimisation, importance des parametres, coordonnees paralleles
optuna_figures = plot_optuna_results(study_xgb)

In [13]:
# === COMPARAISON DES 3 APPROCHES D'OPTIMISATION ===
print("=" * 70)
print("COMPARAISON DES STRATEGIES D'OPTIMISATION (XGBoost)")
print("=" * 70)

comparison_data = {
    "Methode": ["GridSearchCV", "Hyperopt (TPE)", "Optuna (TPE)"],
    "Meilleur F1 (CV)": [best_score_grid, best_f1_xgb, best_score_optuna],
    "Nb essais": [
        len(grid_xgb.cv_results_["mean_test_score"]),
        max_evals,
        max_evals,
    ],
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

# Graphique comparatif
fig_compare = go.Figure(
    data=[
        go.Bar(
            x=df_comparison["Methode"],
            y=df_comparison["Meilleur F1 (CV)"],
            text=[f"{v:.4f}" for v in df_comparison["Meilleur F1 (CV)"]],
            textposition="outside",
            marker_color=["#636EFA", "#EF553B", "#00CC96"],
        )
    ]
)
fig_compare.update_layout(
    title="Comparaison des strategies d'optimisation (XGBoost)",
    yaxis_title="F1 Score (CV)",
    yaxis_range=[0, max(df_comparison["Meilleur F1 (CV)"]) * 1.15],
    width=600,
    height=400,
)
fig_compare.show()

print("\nConclusion :")
best_method = df_comparison.loc[df_comparison["Meilleur F1 (CV)"].idxmax(), "Methode"]
print(f"  Meilleure methode : {best_method}")
print(f"  GridSearchCV est exhaustif mais lent (O(n^k) combinaisons)")
print(f"  Hyperopt et Optuna convergent plus vite avec moins d'essais")

COMPARAISON DES STRATEGIES D'OPTIMISATION (XGBoost)
       Methode  Meilleur F1 (CV)  Nb essais
  GridSearchCV          0.545291         27
Hyperopt (TPE)          0.547808         50
  Optuna (TPE)          0.548424         50



Conclusion :
  Meilleure methode : Optuna (TPE)
  GridSearchCV est exhaustif mais lent (O(n^k) combinaisons)
  Hyperopt et Optuna convergent plus vite avec moins d'essais


## 4. Impact du nombre d'arbres (`n_estimators`)

**Question :** Au-delà de 300 arbres, y a-t-il encore un gain de performance ?

**Attendu :** Les performances devraient se stabiliser au-delà d'un certain seuil (loi des rendements décroissants).

In [14]:
# Test nombre d'arbres sur ACCIDENT - TOUTES LES MÉTRIQUES
n_estimators_list_acc = [20, 50, 100, 200]
results_trees_acc = []

for n_trees in n_estimators_list_acc:
    print(f"Testing n_estimators={n_trees} sur ACCIDENT...", end=" ")

    model_acc = create_pipeline(
        RandomForestClassifier(
            n_estimators=n_trees, random_state=RANDOM_STATE, n_jobs=nb_workers, class_weight="balanced"
        )
    )

    res, y_pred_tree, y_proba_tree = evaluate_model(
        model_acc, X_train_acc_bin, X_test_acc_bin, y_train_acc_bin, y_test_acc_bin, f"RF_{n_trees}"
    )
    res["n_estimators"] = n_trees

    # Métriques supplémentaires
    cm = confusion_matrix(y_test_acc_bin, y_pred_tree)
    tn, fp, fn, tp = cm.ravel()
    res["specificity"] = tn / (tn + fp) if (tn + fp) > 0 else 0
    res["mcc"] = matthews_corrcoef(y_test_acc_bin, y_pred_tree)

    results_trees_acc.append(res)
    print(f"F1={res['f1']:.3f} | Recall={res['recall']:.3f} | AUC={res.get('auc', 0):.3f}")

df_trees_acc = pd.DataFrame(results_trees_acc)
print("\n📊 Tableau complet ACCIDENT:")
print(df_trees_acc[["n_estimators", "accuracy", "precision", "recall", "specificity", "f1", "auc", "mcc"]].round(3))

# Graphique avec toutes les métriques
metrics_to_plot = ["accuracy", "precision", "recall", "specificity", "f1", "auc", "mcc"]
fig = go.Figure()

for metric in metrics_to_plot:
    if metric in df_trees_acc.columns:
        fig.add_trace(
            go.Scatter(
                x=df_trees_acc["n_estimators"],
                y=df_trees_acc[metric],
                mode="lines+markers",
                name=metric.upper(),
                text=[f"{v:.3f}" for v in df_trees_acc[metric]],
                hovertemplate=f"{metric}: %{{y:.3f}}<extra></extra>",
            )
        )

fig.update_layout(
    title="Impact du nombre d'arbres - ACCIDENT (toutes métriques)",
    xaxis_title="Nombre d'arbres (n_estimators)",
    yaxis_title="Score",
    yaxis_range=[0, 1],
    width=900,
    height=500,
    legend_title="Métrique",
)
fig.show()

Testing n_estimators=20 sur ACCIDENT... F1=0.683 | Recall=0.684 | AUC=0.705
Testing n_estimators=50 sur ACCIDENT... F1=0.682 | Recall=0.683 | AUC=0.705
Testing n_estimators=100 sur ACCIDENT... F1=0.683 | Recall=0.683 | AUC=0.705
Testing n_estimators=200 sur ACCIDENT... F1=0.682 | Recall=0.683 | AUC=0.705

📊 Tableau complet ACCIDENT:
   n_estimators  accuracy  precision  recall  specificity     f1    auc    mcc
0            20     0.684      0.682   0.684        0.763  0.683  0.705  0.303
1            50     0.683      0.682   0.683        0.759  0.682  0.705  0.302
2           100     0.683      0.682   0.683        0.760  0.683  0.705  0.302
3           200     0.683      0.682   0.683        0.760  0.682  0.705  0.302


## 5. Comparaison des algorithmes de boosting

**Algorithmes testés :**

| Algorithme | Avantages | Inconvénients |
|------------|-----------|---------------|
| **RandomForest** | Robuste, interprétable, peu d'hyperparamètres | Peut être moins performant |
| **XGBoost** | Très performant, régularisation intégrée | Plus lent à entraîner |
| **LightGBM** | Très rapide, gère bien les grandes données | Peut overfitter |
| **CatBoost** | Gère nativement les catégorielles | Incompatible sklearn Pipeline (1.6+) |

**Objectif :** Identifier l'algorithme avec le meilleur compromis F1/AUC.

In [15]:
# === COMPARAISON DES 6 MODÈLES ACCIDENT ===
print("=" * 70)
print("COMPARAISON DES MODÈLES ACCIDENT (6 modèles)")
print("=" * 70)

models_eval_acc = []
trained_models_acc = {}

# 1. RF_baseline
rf_baseline = create_pipeline(
    RandomForestClassifier(
        n_estimators=base_estimators, random_state=RANDOM_STATE, n_jobs=nb_workers, class_weight="balanced"
    )
)
res, y_pred, y_proba = evaluate_model(
    rf_baseline, X_train_acc_bin, X_test_acc_bin, y_train_acc_bin, y_test_acc_bin, "RF_baseline"
)
models_eval_acc.append({"name": "RF_baseline", "y_true": y_test_acc_bin, "y_pred": y_pred, "y_proba": y_proba})
trained_models_acc["RF_baseline"] = rf_baseline

# 2. XGB_baseline
xgb_baseline = create_pipeline(
    XGBClassifier(
        n_estimators=base_estimators,
        random_state=RANDOM_STATE,
        n_jobs=nb_workers,
        scale_pos_weight=len(y_train_acc_bin[y_train_acc_bin == 0]) / len(y_train_acc_bin[y_train_acc_bin == 1]),
    )
)
res, y_pred, y_proba = evaluate_model(
    xgb_baseline, X_train_acc_bin, X_test_acc_bin, y_train_acc_bin, y_test_acc_bin, "XGB_baseline"
)
models_eval_acc.append({"name": "XGB_baseline", "y_true": y_test_acc_bin, "y_pred": y_pred, "y_proba": y_proba})
trained_models_acc["XGB_baseline"] = xgb_baseline

# 3. RF_HyperOpt (RandomForest optimisé)
rf_hyperopt = create_pipeline(
    RandomForestClassifier(
        n_estimators=best_params_rf.get("n_estimators", 200),
        max_depth=best_params_rf.get("max_depth", None),
        min_samples_split=best_params_rf.get("min_samples_split", 2),
        random_state=RANDOM_STATE,
        n_jobs=nb_workers,
        class_weight="balanced",
    )
)
res, y_pred, y_proba = evaluate_model(
    rf_hyperopt, X_train_acc_bin, X_test_acc_bin, y_train_acc_bin, y_test_acc_bin, "RF_HyperOpt"
)
models_eval_acc.append({"name": "RF_HyperOpt", "y_true": y_test_acc_bin, "y_pred": y_pred, "y_proba": y_proba})
trained_models_acc["RF_HyperOpt"] = rf_hyperopt

# 4. XGBoost optimisé
xgb_opt = create_pipeline(
    XGBClassifier(
        **best_params_xgb,
        random_state=RANDOM_STATE,
        n_jobs=nb_workers,
        scale_pos_weight=len(y_train_acc_bin[y_train_acc_bin == 0]) / len(y_train_acc_bin[y_train_acc_bin == 1]),
    )
)
res, y_pred, y_proba = evaluate_model(
    xgb_opt, X_train_acc_bin, X_test_acc_bin, y_train_acc_bin, y_test_acc_bin, "XGBoost"
)
models_eval_acc.append({"name": "XGBoost", "y_true": y_test_acc_bin, "y_pred": y_pred, "y_proba": y_proba})
trained_models_acc["XGBoost"] = xgb_opt

# 5. LightGBM optimisé
lgbm_opt = create_pipeline(
    LGBMClassifier(
        **best_params_lgbm, random_state=RANDOM_STATE, n_jobs=nb_workers, class_weight="balanced", verbose=-1
    )
)
res, y_pred, y_proba = evaluate_model(
    lgbm_opt, X_train_acc_bin, X_test_acc_bin, y_train_acc_bin, y_test_acc_bin, "LightGBM"
)
models_eval_acc.append({"name": "LightGBM", "y_true": y_test_acc_bin, "y_pred": y_pred, "y_proba": y_proba})
trained_models_acc["LightGBM"] = lgbm_opt

# 6. CatBoost optimisé (sans Pipeline - incompatible sklearn 1.6+)
catboost_opt = CatBoostClassifier(
    **best_params_cb, random_state=RANDOM_STATE, auto_class_weights="Balanced", verbose=False, allow_writing_files=False
)
res, y_pred, y_proba = evaluate_catboost(
    catboost_opt, X_train_acc_bin, X_test_acc_bin, y_train_acc_bin, y_test_acc_bin, "CatBoost"
)
models_eval_acc.append({"name": "CatBoost", "y_true": y_test_acc_bin, "y_pred": y_pred, "y_proba": y_proba})
trained_models_acc["CatBoost"] = catboost_opt

# Affichage complet avec display_metrics (confusion matrices, ROC, bar chart, table)
eval_results = display_metrics(models_results=models_eval_acc, class_labels=["Non grave", "Grave"])
figures_tuning = eval_results["figures"]

COMPARAISON DES MODÈLES ACCIDENT (6 modèles)


d:\Suivi_de_formation\simplon-ai-developer-training\W31-W33-ML-DEPLOYMENT-CI-CD\mlflow-EDUCATIONAL\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

d:\Suivi_de_formation\simplon-ai-developer-training\W31-W33-ML-DEPLOYMENT-CI-CD\mlflow-EDUCATIONAL\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



EVALUATION DES MODELES (6 modele(s))

📊 TABLEAU DES METRIQUES
----------------------------------------------------------------------
       model  accuracy  balanced_accuracy  precision  recall     f1  roc_auc    mcc  specificity  sensitivity
 RF_baseline    0.6831             0.6505     0.6819  0.6831 0.6825   0.7055 0.3021       0.7601       0.5409
XGB_baseline    0.6874             0.6521     0.6844  0.6874 0.6858   0.7087 0.3075       0.7706       0.5336
 RF_HyperOpt    0.6564             0.6277     0.6596  0.6564 0.6579   0.6924 0.2531       0.7243       0.5310
     XGBoost    0.6890             0.6532     0.6857  0.6890 0.6872   0.7093 0.3102       0.7735       0.5329
    LightGBM    0.6868             0.6520     0.6842  0.6868 0.6854   0.7086 0.3069       0.7691       0.5349
    CatBoost    0.6904             0.6525     0.6859  0.6904 0.6877   0.7091 0.3103       0.7800       0.5249




📈 MATRICES DE CONFUSION
----------------------------------------------------------------------



📉 COURBES ROC
----------------------------------------------------------------------



RESUME
Meilleur modele (F1): CatBoost (F1 = 0.6877)
Meilleur modele (AUC): XGBoost (AUC = 0.7093)


In [16]:
# === Log des modèles fine-tunés dans MLflow ===
if mlflow_ready:
    for model_info in models_eval_acc:
        name = model_info["name"]
        model = trained_models_acc[name]

        # Métriques depuis le DataFrame retourné par display_metrics
        metrics_df = eval_results["metrics_df"]
        row = metrics_df[metrics_df["model"] == name].iloc[0]

        metrics = {
            "accuracy": row.get("accuracy"),
            "precision": row.get("precision"),
            "recall": row.get("recall"),
            "f1": row.get("f1"),
            "auc": row.get("roc_auc"),
        }

        log_training_run(
            run_name=f"tuning_{name}",
            model=model,
            model_name=name,
            params={k: str(v) for k, v in model.get_params().items() if v is not None},
            metrics=metrics,
            y_true=model_info["y_true"],
            y_pred=model_info["y_pred"],
            y_proba=model_info["y_proba"],
            class_labels=["Non grave", "Grave"],
            figures=figures_tuning,
            tags={"stage": "tuning", "target": "grav_binary"},
            registered_model_name=f"tuning-{name}",
        )

2026/02/25 11:45:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'tuning-RF_baseline' already exists. Creating a new version of this model...
2026/02/25 11:45:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tuning-RF_baseline, version 2
Created version '2' of model 'tuning-RF_baseline'.


  MLflow: run 'tuning_RF_baseline' logue (id: dac104a1)
🏃 View run tuning_RF_baseline 25/02/2026 11h:44m:59s at: http://localhost:5000/#/experiments/3/runs/dac104a14848428cbf757bbb533e59fc
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/02/25 11:45:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'tuning-XGB_baseline' already exists. Creating a new version of this model...
2026/02/25 11:45:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tuning-XGB_baseline, version 2
Created version '2' of model 'tuning-XGB_baseline'.


  MLflow: run 'tuning_XGB_baseline' logue (id: 0736bfdc)
🏃 View run tuning_XGB_baseline 25/02/2026 11h:45m:19s at: http://localhost:5000/#/experiments/3/runs/0736bfdcdd1f4657a0b81d986e2b1f50
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/02/25 11:45:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'tuning-RF_HyperOpt' already exists. Creating a new version of this model...
2026/02/25 11:45:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tuning-RF_HyperOpt, version 2
Created version '2' of model 'tuning-RF_HyperOpt'.


...

  MLflow: run 'tuning_XGBoost' logue (id: 703b9d12)
🏃 View run tuning_XGBoost 25/02/2026 11h:45m:53s at: http://localhost:5000/#/experiments/3/runs/703b9d12cd774891b91eed052ae58e2a
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/02/25 11:46:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'tuning-LightGBM' already exists. Creating a new version of this model...
2026/02/25 11:46:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tuning-LightGBM, version 2
Created version '2' of model 'tuning-LightGBM'.


  MLflow: run 'tuning_LightGBM' logue (id: b120344e)
🏃 View run tuning_LightGBM 25/02/2026 11h:46m:12s at: http://localhost:5000/#/experiments/3/runs/b120344e41e3464c8bd961e1a4d62f8c
🧪 View experiment at: http://localhost:5000/#/experiments/3


Registered model 'tuning-CatBoost' already exists. Creating a new version of this model...
2026/02/25 11:46:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tuning-CatBoost, version 2
Created version '2' of model 'tuning-CatBoost'.


  MLflow: run 'tuning_CatBoost' logue (id: ca1dabae)
🏃 View run tuning_CatBoost 25/02/2026 11h:46m:31s at: http://localhost:5000/#/experiments/3/runs/ca1dabaea8464788a7e2ded831c3e16f
🧪 View experiment at: http://localhost:5000/#/experiments/3


## 6. Comparaison finale et sélection du meilleur modèle

**Critères de sélection :**
1. **F1-Score** (critère principal)
2. **AUC** (départage en cas d'égalité sur F1)

**Modèles comparés :**
- RandomForest baseline
- RandomForest optimisé (meilleurs hyperparamètres)
- RandomForest avec plus d'arbres
- XGBoost
- LightGBM
- CatBoost

In [17]:
# Sélection du meilleur modèle ACCIDENT
print("=" * 70)
print("SÉLECTION DU MEILLEUR MODÈLE ACCIDENT")
print("=" * 70)

best_model_info = select_best_model(models_eval_acc, metric="f1")
print(f"\nMeilleur modèle: {best_model_info['name']}")
print(f"F1-Score: {best_model_info['score']:.4f}")

SÉLECTION DU MEILLEUR MODÈLE ACCIDENT

Meilleur modèle: CatBoost
F1-Score: 0.6877


### 6.1 Interprétation des matrices de confusion

**Lecture de la matrice :**
- **TP (True Positive)** : Accidents graves correctement détectés → Bon !
- **FN (False Negative)** : Accidents graves manqués → Dangereux (sous-estimation du risque)
- **FP (False Positive)** : Fausses alertes → Coûteux mais pas dangereux
- **TN (True Negative)** : Non-graves correctement identifiés → Bon !

**Compromis à faire :**
- Privilégier le **Recall** si on veut minimiser les FN (ne pas rater de cas graves)
- Privilégier la **Precision** si on veut minimiser les FP (éviter les fausses alertes)

## 7. Sauvegarde modeles optimises

In [18]:
# Sauvegarder le meilleur modèle ACCIDENT
print("=" * 70)
print("SAUVEGARDE DU MEILLEUR MODÈLE ACCIDENT")
print("=" * 70)

# Configuration des modèles (même config que la comparaison)
X_acc_bin_full, y_acc_bin_full = prepare_data(df_accident, "grav_binary")
pos_weight = len(y_acc_bin_full[y_acc_bin_full == 0]) / len(y_acc_bin_full[y_acc_bin_full == 1])

model_configs = {
    "RF_baseline": lambda: create_pipeline(
        RandomForestClassifier(
            n_estimators=base_estimators, random_state=RANDOM_STATE, n_jobs=nb_workers, class_weight="balanced"
        )
    ),
    "XGB_baseline": lambda: create_pipeline(
        XGBClassifier(
            n_estimators=base_estimators, random_state=RANDOM_STATE, n_jobs=nb_workers, scale_pos_weight=pos_weight
        )
    ),
    "RF_HyperOpt": lambda: create_pipeline(
        RandomForestClassifier(
            n_estimators=best_params_rf.get("n_estimators", 200),
            max_depth=best_params_rf.get("max_depth", None),
            min_samples_split=best_params_rf.get("min_samples_split", 2),
            random_state=RANDOM_STATE,
            n_jobs=nb_workers,
            class_weight="balanced",
        )
    ),
    "XGBoost": lambda: create_pipeline(
        XGBClassifier(**best_params_xgb, random_state=RANDOM_STATE, n_jobs=nb_workers, scale_pos_weight=pos_weight)
    ),
    "LightGBM": lambda: create_pipeline(
        LGBMClassifier(
            **best_params_lgbm, random_state=RANDOM_STATE, n_jobs=nb_workers, class_weight="balanced", verbose=-1
        )
    ),
    "CatBoost": lambda: CatBoostClassifier(
        **best_params_cb,
        random_state=RANDOM_STATE,
        auto_class_weights="Balanced",
        verbose=False,
        allow_writing_files=False,
    ),
}

result = save_best_model(
    best_model_name=best_model_info["name"],
    model_configs=model_configs,
    X_full=X_acc_bin_full,
    y_full=y_acc_bin_full,
    X_test=X_test_acc_bin,
    y_test=y_test_acc_bin,
    save_path="models/model_accident_binary_optimized.joblib",
)

SAUVEGARDE DU MEILLEUR MODÈLE ACCIDENT
Entrainement du modele: CatBoost
Sauvegarde: models/model_accident_binary_optimized.joblib
F1 sur test (weighted): 0.6905

Features attendues en entree (9):
  1. est_nuit
  2. est_heure_pointe
  3. jour_semaine
  4. est_weekend
  5. agg
  6. vma
  7. impl_vehicule_leger
  8. impl_poids_lourd
  9. impl_pieton


## 8. Résumé et conclusions

### Modèles sauvegardés

| Fichier | Description | Usage |
|---------|-------------|-------|
|  | Classification binaire - baseline | Fallback |
|  | Binaire optimisé (Hyperopt) | **Production** |

### Résultats de l'optimisation

| Algorithme | F1 CV (Hyperopt) | F1 Test | Commentaire |
|------------|------------------|---------|-------------|
| RandomForest | ~0.64 | ~0.53 | Overfitting possible |
| XGBoost | ~0.65 | ~0.66 | Bon compromis |
| LightGBM | ~0.60 | ~0.66 | Rapide et performant |
| CatBoost | - | ~0.66 | **Sélectionné** |

### Recommandations pour l'API

1. **Endpoint principal** : Utiliser 
2. **Preprocessing** : Appliquer le même pipeline (imputation + scaling) qu'à l'entraînement
3. **Seuil de décision** : Par défaut 0.5, ajustable selon le compromis FP/FN souhaité

### Pistes d'amélioration future

- Enrichir les features avec des données externes (météo détaillée, trafic)
- Tester d'autres architectures (stacking, neural networks)
- Optimiser le seuil de décision selon les coûts métier (FN plus grave que FP)